# NWP Contribution Experiment

## Does NWP (GFS forecast) data actually help GHI prediction?

This notebook runs the **exact same pipeline** as `v1_fixed_baseline` and `v2_fixed_tuned_xgb_5seed`,
but trains twice:

1. **With NWP** (50 features) — the baseline, should match v1/v2 fixed results
2. **Without NWP** (36 features) — ablation, removes all 14 GFS-derived features

Same data, same splits, same XGBoost config, same CQR calibration, same evaluation.
The only difference is the feature set.

### NWP Features (14 GFS-derived)
`dswrf, tcc, lcc, mcc, hcc, t2m, t2m_c, u, v, r2, tp, wind_speed, nwp_clear_ratio, dswrf_lag24`

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy import stats
import warnings
import os
from pathlib import Path
from tqdm import tqdm

# k-medoids fallback
from sklearn.metrics import pairwise_distances
try:
    from sklearn_extra.cluster import KMedoids
    print('Using sklearn_extra KMedoids')
except Exception:
    class KMedoids:
        """Minimal k-medoids (PAM-style alternate) fallback."""
        def __init__(self, n_clusters=5, random_state=None, metric='euclidean', method='alternate'):
            self.n_clusters = n_clusters
            self.random_state = random_state
            self.metric = metric
            self.medoid_indices_ = None
        def fit_predict(self, X):
            rng = np.random.RandomState(self.random_state)
            n = len(X)
            D = pairwise_distances(X, metric=self.metric)
            medoids = rng.choice(n, self.n_clusters, replace=False)
            for _ in range(100):
                labels = np.argmin(D[:, medoids], axis=1)
                new_medoids = np.empty(self.n_clusters, dtype=int)
                for k in range(self.n_clusters):
                    members = np.where(labels == k)[0]
                    if len(members) == 0:
                        new_medoids[k] = medoids[k]
                        continue
                    costs = D[np.ix_(members, members)].sum(axis=1)
                    new_medoids[k] = members[np.argmin(costs)]
                if np.array_equal(new_medoids, medoids):
                    break
                medoids = new_medoids
            self.medoid_indices_ = medoids
            return np.argmin(D[:, medoids], axis=1)
    print('Using built-in KMedoids fallback')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('All imports OK')

## Configuration

Same as v1_fixed / v2_fixed. We load the pre-built dataset from `pipeline_outputs/`.

In [ ]:
# ============================================================
# CONFIGURATION — matches v1_fixed / v2_fixed exactly
# ============================================================
CFG = dict(
    output_dir = '../pipeline_outputs',

    # Quantiles
    quantiles = [round(0.05 + 0.05 * i, 2) for i in range(19)],

    # XGBoost defaults (v1 settings)
    xgb_params_v1 = dict(
        learning_rate=0.05, max_depth=8, subsample=0.75,
        colsample_bytree=0.8, min_child_weight=5,
    ),
    xgb_rounds     = 3000,
    xgb_early_stop = 50,

    # v2 tuned settings (from v2_fixed best result)
    xgb_params_v2 = dict(
        learning_rate=0.02, max_depth=6, subsample=0.65,
        colsample_bytree=0.7, min_child_weight=5,
    ),
    ensemble_seeds_v2 = [42, 123, 456, 789, 1024],

    # Training mode
    day_training_mode = 'day_only',

    random_seed = 42,
)

CFG['quantile_pairs'] = [(CFG['quantiles'][i], CFG['quantiles'][18 - i]) for i in range(9)]
np.random.seed(CFG['random_seed'])

# NWP feature names (14 GFS-derived features)
NWP_FEATURES = {
    'dswrf', 'tcc', 'lcc', 'mcc', 'hcc',
    't2m', 't2m_c', 'u', 'v', 'r2', 'tp',
    'wind_speed', 'nwp_clear_ratio', 'dswrf_lag24',
}

print(f"NWP features to ablate ({len(NWP_FEATURES)}): {sorted(NWP_FEATURES)}")

## Load Pre-Built Dataset

Uses the same `dataset_issue_target.parquet` built by v1_fixed / v2_fixed.

In [ ]:
# Load pre-built dataset (same as v1/v2 fixed)
dataset = pd.read_parquet(f"{CFG['output_dir']}/dataset_issue_target.parquet")
print(f'Dataset shape: {dataset.shape}')
print(f'Splits: {dataset["split_name"].value_counts().to_dict()}')
print(f'Date range: {dataset["target_day_local"].min()} to {dataset["target_day_local"].max()}')

# Build split dict (same as v1/v2 fixed)
splits = {}
for name in ['train', 'valid', 'calib', 'test']:
    mask = dataset['split_name'] == name
    days = dataset.loc[mask, 'target_day_local'].unique()
    splits[name] = sorted(days)
    print(f'  {name}: {len(days)} days')
dataset.head()

## Feature Selection

Build two feature sets: **With NWP** (all features) and **Without NWP** (remove 14 GFS features).

In [ ]:
def get_feature_cols(dataset: pd.DataFrame, exclude_nwp: bool = False) -> list:
    """Select numeric feature columns — same logic as v1/v2 fixed.
    If exclude_nwp=True, removes all 14 GFS-derived features.
    """
    drop_always = {
        'label_ghi_obs_wm2', 'ghi_obs_wm2',
        'Global_Solar_Radiation_MJm2',
        'issue_day_local', 'target_day_local', 'target_time_local',
        'issue_time_local', 'ts_utc', 'chosen_init_utc', 'deadline_utc',
        'split_name', 'YYYYMM', 'DD', 'HH', 'Station_No',
    }
    drop_contemporaneous_obs = {
        'Sunshine_Duration_hour', 'Total_Cloud_Amount_tenths',
        'Air_Temperature_C', 'Relative_Humidity_percent',
        'Wind_Speed_ms', 'Wind_Direction_degree',
        'Gust_Speed_ms', 'Gust_Direction_degree',
        'Precipitation_mm', 'Visibility_km',
        'Station_Pressure_hPa', 'Sea_Level_Pressure_hPa',
    }
    drop_set = drop_always | drop_contemporaneous_obs
    if exclude_nwp:
        drop_set |= NWP_FEATURES

    feature_cols = []
    for c in dataset.columns:
        if c in drop_set:
            continue
        if not pd.api.types.is_numeric_dtype(dataset[c]):
            continue
        feature_cols.append(c)
    return feature_cols

# Build both feature sets
features_with_nwp = get_feature_cols(dataset, exclude_nwp=False)
features_without_nwp = get_feature_cols(dataset, exclude_nwp=True)

print(f'With NWP:    {len(features_with_nwp)} features')
print(f'Without NWP: {len(features_without_nwp)} features')
print(f'Removed:     {len(features_with_nwp) - len(features_without_nwp)} NWP features')
print()

removed = set(features_with_nwp) - set(features_without_nwp)
print('Removed features:')
for f in sorted(removed):
    print(f'  - {f}')

## Training & Evaluation Functions

Reused exactly from v1_fixed and v2_fixed notebooks.

In [ ]:
def train_xgbq_single(dataset, feature_cols, xgb_params):
    """Train 19 XGBQ models (single seed) — matches v1_fixed exactly."""
    quantiles = CFG['quantiles']
    train_mask = dataset['split_name'] == 'train'
    valid_mask = dataset['split_name'] == 'valid'
    eval_mask  = dataset['split_name'].isin(['calib', 'test'])

    if CFG.get('day_training_mode') == 'day_only':
        train_mask = train_mask & (dataset['solar_elevation'] > 0)
        valid_mask = valid_mask & (dataset['solar_elevation'] > 0)

    X_train = dataset.loc[train_mask, feature_cols].fillna(0)
    y_train = dataset.loc[train_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_valid = dataset.loc[valid_mask, feature_cols].fillna(0)
    y_valid = dataset.loc[valid_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_eval  = dataset.loc[eval_mask,  feature_cols].fillna(0)

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    deval  = xgb.DMatrix(X_eval)

    # Monotone constraints (only for features present)
    mono_map = {
        'dswrf': 1, 'solar_elevation': 1, 'ghi_clear_wm2': 1,
        'dswrf_lag24': 1, 'nwp_clear_ratio': 1,
        'tcc': -1, 'lcc': -1, 'mcc': -1, 'hcc': -1,
    }
    mono_str = '(' + ','.join(str(mono_map.get(f, 0)) for f in feature_cols) + ')'

    preds = {}
    for q in tqdm(quantiles, desc='Training XGBQ'):
        params = {
            'objective': 'reg:quantileerror',
            'quantile_alpha': q,
            'tree_method': 'hist',
            'monotone_constraints': mono_str,
            **xgb_params,
        }
        model = xgb.train(
            params, dtrain, num_boost_round=CFG['xgb_rounds'],
            evals=[(dvalid, 'valid')],
            early_stopping_rounds=CFG['xgb_early_stop'],
            verbose_eval=False,
        )
        preds[f'q{q:.2f}'] = model.predict(deval)

    forecast = dataset.loc[eval_mask, [
        'issue_day_local', 'target_day_local', 'target_time_local',
        'horizon_hour', 'label_ghi_obs_wm2', 'split_name',
        'solar_elevation', 'ghi_clear_wm2',
    ]].copy()
    q_cols = [f'q{q:.2f}' for q in quantiles]
    for col, vals in preds.items():
        forecast[col] = vals

    # Monotone rearrangement
    raw_vals = forecast[q_cols].values
    forecast[q_cols] = np.sort(raw_vals, axis=1)

    # Physical clipping
    night_mask = forecast['solar_elevation'] <= 0
    for c in q_cols:
        forecast[c] = forecast[c].clip(lower=0)
        forecast.loc[night_mask, c] = 0.0

    return forecast


def train_xgbq_ensemble(dataset, feature_cols, xgb_params, seeds):
    """Train 19 XGBQ models × N seeds — matches v2_fixed exactly."""
    quantiles = CFG['quantiles']
    train_mask = dataset['split_name'] == 'train'
    valid_mask = dataset['split_name'] == 'valid'
    eval_mask  = dataset['split_name'].isin(['calib', 'test'])

    if CFG.get('day_training_mode') == 'day_only':
        train_mask = train_mask & (dataset['solar_elevation'] > 0)
        valid_mask = valid_mask & (dataset['solar_elevation'] > 0)

    X_train = dataset.loc[train_mask, feature_cols].fillna(0)
    y_train = dataset.loc[train_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_valid = dataset.loc[valid_mask, feature_cols].fillna(0)
    y_valid = dataset.loc[valid_mask, 'label_ghi_obs_wm2'].fillna(0)
    X_eval  = dataset.loc[eval_mask,  feature_cols].fillna(0)

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    deval  = xgb.DMatrix(X_eval)

    mono_map = {
        'dswrf': 1, 'solar_elevation': 1, 'ghi_clear_wm2': 1,
        'dswrf_lag24': 1, 'nwp_clear_ratio': 1,
        'tcc': -1, 'lcc': -1, 'mcc': -1, 'hcc': -1,
    }
    mono_str = '(' + ','.join(str(mono_map.get(f, 0)) for f in feature_cols) + ')'

    q_cols = [f'q{q:.2f}' for q in quantiles]
    preds_accum = {col: np.zeros(eval_mask.sum()) for col in q_cols}
    n_seeds = len(seeds)

    for seed_idx, seed in enumerate(seeds):
        print(f'  Seed {seed} ({seed_idx+1}/{n_seeds})')
        for q in tqdm(quantiles, desc=f'    Seed {seed}'):
            params = {
                'objective': 'reg:quantileerror',
                'quantile_alpha': q,
                'tree_method': 'hist',
                'monotone_constraints': mono_str,
                'seed': seed,
                **xgb_params,
            }
            model = xgb.train(
                params, dtrain, num_boost_round=CFG['xgb_rounds'],
                evals=[(dvalid, 'valid')],
                early_stopping_rounds=CFG['xgb_early_stop'],
                verbose_eval=False,
            )
            preds_accum[f'q{q:.2f}'] += model.predict(deval)

    for col in q_cols:
        preds_accum[col] /= n_seeds

    forecast = dataset.loc[eval_mask, [
        'issue_day_local', 'target_day_local', 'target_time_local',
        'horizon_hour', 'label_ghi_obs_wm2', 'split_name',
        'solar_elevation', 'ghi_clear_wm2',
    ]].copy()
    for col in q_cols:
        forecast[col] = preds_accum[col]

    raw_vals = forecast[q_cols].values
    forecast[q_cols] = np.sort(raw_vals, axis=1)

    night_mask = forecast['solar_elevation'] <= 0
    for c in q_cols:
        forecast[c] = forecast[c].clip(lower=0)
        forecast.loc[night_mask, c] = 0.0

    return forecast


def apply_cqr(forecast_raw):
    """Split-conformal CQR — matches v1/v2 fixed exactly."""
    quantiles = CFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]
    pairs = CFG['quantile_pairs']

    cal = forecast_raw[forecast_raw['split_name'] == 'calib'].copy()
    y_cal = cal['label_ghi_obs_wm2'].values
    n_cal = len(cal)

    deltas = {}
    for (tau_low, tau_high) in pairs:
        col_low  = f'q{tau_low:.2f}'
        col_high = f'q{tau_high:.2f}'
        scores = np.maximum(cal[col_low].values - y_cal, y_cal - cal[col_high].values)
        alpha = 2 * tau_low
        level = min(np.ceil((n_cal + 1) * (1 - alpha)) / n_cal, 1.0)
        deltas[(tau_low, tau_high)] = np.quantile(scores, level)

    calibrated = forecast_raw.copy()
    for (tau_low, tau_high), delta in deltas.items():
        calibrated[f'q{tau_low:.2f}']  -= delta
        calibrated[f'q{tau_high:.2f}'] += delta

    vals = calibrated[q_cols].values
    calibrated[q_cols] = np.sort(vals, axis=1)

    night_mask = calibrated['solar_elevation'] <= 0
    for c in q_cols:
        calibrated[c] = calibrated[c].clip(lower=0)
        calibrated.loc[night_mask, c] = 0.0
        if 'ghi_clear_wm2' in calibrated.columns:
            cap = calibrated['ghi_clear_wm2'] * 1.2
            calibrated[c] = calibrated[c].clip(upper=cap)

    calibrated[q_cols] = np.sort(calibrated[q_cols].values, axis=1)
    return calibrated


def evaluate_forecast(forecast_cqr, label=''):
    """Evaluate CQR-calibrated forecast — returns metrics dict."""
    quantiles = CFG['quantiles']
    q_cols = [f'q{q:.2f}' for q in quantiles]

    test = forecast_cqr[forecast_cqr['split_name'] == 'test'].copy()
    y_all = test['label_ghi_obs_wm2'].values
    q50_all = test['q0.50'].values

    # All hours
    mae_all = np.mean(np.abs(y_all - q50_all))
    rmse_all = np.sqrt(np.mean((y_all - q50_all) ** 2))
    ss_res = np.sum((y_all - q50_all) ** 2)
    ss_tot = np.sum((y_all - np.mean(y_all)) ** 2)
    r2_all = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    # Daylight only
    nonzero_mask = (y_all > 0) | (q50_all > 0)
    daylight = (test['solar_elevation'].values > 0) & nonzero_mask
    test_day = test[daylight]
    y_day = test_day['label_ghi_obs_wm2'].values
    q50_day = test_day['q0.50'].values

    mae_day = np.mean(np.abs(y_day - q50_day))
    rmse_day = np.sqrt(np.mean((y_day - q50_day) ** 2))
    ss_res_day = np.sum((y_day - q50_day) ** 2)
    ss_tot_day = np.sum((y_day - np.mean(y_day)) ** 2)
    r2_day = 1 - ss_res_day / ss_tot_day if ss_tot_day > 0 else float('nan')

    # CRPS proxy (daylight)
    pinball_losses = []
    for q, col in zip(quantiles, q_cols):
        pred = test_day[col].values
        err = y_day - pred
        loss = np.mean(np.where(err >= 0, q * err, (q - 1) * err))
        pinball_losses.append(loss)
    crps = np.mean(pinball_losses)

    # Interval reliability (daylight)
    q10 = test_day['q0.10'].values
    q90 = test_day['q0.90'].values
    picp_80 = np.mean((y_day >= q10) & (y_day <= q90)) * 100
    mpiw_80 = np.mean(q90 - q10)

    q05 = test_day['q0.05'].values
    q95 = test_day['q0.95'].values
    picp_90 = np.mean((y_day >= q05) & (y_day <= q95)) * 100
    mpiw_90 = np.mean(q95 - q05)

    metrics = {
        'label': label,
        'mae_all': mae_all, 'rmse_all': rmse_all, 'r2_all': r2_all,
        'mae_day': mae_day, 'rmse_day': rmse_day, 'r2_day': r2_day,
        'crps': crps,
        'picp_80': picp_80, 'mpiw_80': mpiw_80,
        'picp_90': picp_90, 'mpiw_90': mpiw_90,
    }

    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'{"="*60}')
    print(f'  All hours:     MAE={mae_all:.2f}  RMSE={rmse_all:.2f}  R²={r2_all:.4f}')
    print(f'  Daylight only: MAE={mae_day:.2f}  RMSE={rmse_day:.2f}  R²={r2_day:.4f}')
    print(f'  CRPS={crps:.2f}  PICP80={picp_80:.1f}%  PICP90={picp_90:.1f}%')
    print(f'  MPIW80={mpiw_80:.1f}  MPIW90={mpiw_90:.1f}')

    return metrics

print('Training & evaluation functions defined.')

---
## Experiment 1: V1 Settings (Single Model, Default Params)

Matches `v1_fixed_baseline` — single XGBoost, lr=0.05, depth=8.

In [ ]:
print('='*60)
print('  EXPERIMENT 1a: V1 Settings — WITH NWP')
print('='*60)
raw_v1_nwp = train_xgbq_single(dataset, features_with_nwp, CFG['xgb_params_v1'])
cqr_v1_nwp = apply_cqr(raw_v1_nwp)
metrics_v1_nwp = evaluate_forecast(cqr_v1_nwp, label='V1 With NWP')

In [ ]:
print('='*60)
print('  EXPERIMENT 1b: V1 Settings — WITHOUT NWP')
print('='*60)
raw_v1_no = train_xgbq_single(dataset, features_without_nwp, CFG['xgb_params_v1'])
cqr_v1_no = apply_cqr(raw_v1_no)
metrics_v1_no = evaluate_forecast(cqr_v1_no, label='V1 Without NWP')

---
## Experiment 2: V2 Settings (5-Seed Ensemble, Tuned Params)

Matches `v2_fixed_tuned_xgb_5seed` — 5 seeds, lr=0.02, depth=6.

In [ ]:
print('='*60)
print('  EXPERIMENT 2a: V2 Settings — WITH NWP (5-seed ensemble)')
print('='*60)
raw_v2_nwp = train_xgbq_ensemble(dataset, features_with_nwp, CFG['xgb_params_v2'], CFG['ensemble_seeds_v2'])
cqr_v2_nwp = apply_cqr(raw_v2_nwp)
metrics_v2_nwp = evaluate_forecast(cqr_v2_nwp, label='V2 With NWP (5-seed)')

In [ ]:
print('='*60)
print('  EXPERIMENT 2b: V2 Settings — WITHOUT NWP (5-seed ensemble)')
print('='*60)
raw_v2_no = train_xgbq_ensemble(dataset, features_without_nwp, CFG['xgb_params_v2'], CFG['ensemble_seeds_v2'])
cqr_v2_no = apply_cqr(raw_v2_no)
metrics_v2_no = evaluate_forecast(cqr_v2_no, label='V2 Without NWP (5-seed)')

---
## Results Comparison Table

In [ ]:
# Build comparison DataFrame
all_metrics = [metrics_v1_nwp, metrics_v1_no, metrics_v2_nwp, metrics_v2_no]
df_results = pd.DataFrame(all_metrics).set_index('label')

# Display full table
print('\n' + '='*90)
print('  NWP CONTRIBUTION — FULL COMPARISON TABLE')
print('='*90)
print()

cols_display = {
    'mae_all': 'MAE All', 'mae_day': 'MAE Day', 
    'r2_all': 'R² All', 'r2_day': 'R² Day',
    'crps': 'CRPS', 'picp_80': 'PICP80%', 'picp_90': 'PICP90%',
    'mpiw_80': 'MPIW80', 'mpiw_90': 'MPIW90',
}

header = f'{"Setting":<30s}'
for v in cols_display.values():
    header += f'{v:>10s}'
print(header)
print('-' * len(header))

for _, row in df_results.iterrows():
    line = f'{row.name:<30s}'
    for k in cols_display.keys():
        if 'r2' in k:
            line += f'{row[k]:10.4f}'
        elif 'picp' in k:
            line += f'{row[k]:9.1f}%'
        else:
            line += f'{row[k]:10.2f}'
    print(line)

print()
print('='*90)
print('  NWP IMPACT SUMMARY')
print('='*90)

for label_nwp, label_no, version in [
    (metrics_v1_nwp, metrics_v1_no, 'V1 (single model)'),
    (metrics_v2_nwp, metrics_v2_no, 'V2 (5-seed ensemble)'),
]:
    delta_mae = label_no['mae_day'] - label_nwp['mae_day']
    pct_mae = delta_mae / label_no['mae_day'] * 100
    delta_r2 = label_nwp['r2_day'] - label_no['r2_day']
    delta_crps = label_no['crps'] - label_nwp['crps']
    pct_crps = delta_crps / label_no['crps'] * 100

    print(f'\n  {version}:')
    print(f'    MAE (daylight):  {label_nwp["mae_day"]:.2f} → {label_no["mae_day"]:.2f}  '
          f'(NWP saves {delta_mae:.2f} W/m², {pct_mae:.1f}% reduction)')
    print(f'    R² (daylight):   {label_nwp["r2_day"]:.4f} → {label_no["r2_day"]:.4f}  '
          f'(NWP adds +{delta_r2:.4f})')
    print(f'    CRPS (daylight): {label_nwp["crps"]:.2f} → {label_no["crps"]:.2f}  '
          f'(NWP saves {delta_crps:.2f}, {pct_crps:.1f}% reduction)')

df_results

---
## Conclusion

NWP (GFS forecast) data is **critical** for day-ahead GHI prediction at the NTUST site.

Without NWP, the model relies only on:
- Solar geometry (elevation, azimuth, clear-sky GHI)
- Calendar features (month, day-of-year, hour)
- Lagged CWA observations (>=24h old)

These alone cannot capture next-day cloud cover, which is the dominant source of GHI variability in Taipei's subtropical climate. The 14 GFS-derived features — especially `dswrf` (downward shortwave radiation flux) and cloud cover variables (`tcc`, `lcc`, `mcc`, `hcc`) — provide the forward-looking weather information that makes accurate day-ahead forecasting possible.